# Comparación aleatoria y grupos ambientales

**Objetivo:** Grupos por elevación y temperatura; NO es validación geográfica.

Este cuaderno contiene el código completo del experimento `train_spatial_validation.py`,
dividido en etapas y acompañado de explicaciones. No necesita ZIP, clonar un
repositorio ni ejecutar un script oculto. Ejecuta las celdas en orden.

**Recorrido:** entorno → carga e inspección → preparación → modelo o análisis →
evaluación → gráficos → descarga y colaboración.

**Entradas:** `dataset_entrenamiento_completo.csv`.

Los datos no se incluyen. Se conservan los valores predeterminados del protocolo histórico.
Las salidas se generarán al ejecutar; las figuras históricas no están incrustadas.


**Límite del experimento:** este código histórico usa quintiles de elevación + temperatura × 100 y retiene un solo grupo. No es validación espacial ni validación cruzada por provincias. Los nombres geográficos de las variables heredadas no cambian esa limitación.

## 1. Entorno y parámetros
Modifica los parámetros aquí para que una colaboración sea fácil de revisar. Los valores originales siguen seleccionados. Cada ejecución crea una carpeta nueva.

In [ ]:
# Instala únicamente las bibliotecas que falten en el entorno.
import importlib.util
import importlib.metadata
import subprocess
import sys
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image

DEPENDENCIAS = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'torch': 'torch'}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *faltantes], check=True)

CONFIG = dict(seed=42, rf_trees=100, rf_depth=15, epochs=30,
              learning_rate=0.002, batch_size=256)
EXPERIMENTO = 'bloques_ambientales'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATA_DIR = Path(globals().get('DATA_DIR_OVERRIDE', 'data')).resolve()
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR_OVERRIDE', Path('results') / EXPERIMENTO / RUN_ID)).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
print('Parámetros:', CONFIG)
print('Salidas:', OUTPUT_DIR)

def mostrar_figura(ruta):
    # Conserva la figura en disco y también la muestra en el cuaderno.
    if str(ruta).lower().endswith('.png'):
        display(Image(filename=str(ruta)))


## 2. Carga e inspección de los datos
En Colab, la siguiente celda ofrece el selector si faltan archivos. Carga solo los CSV indicados, conservando sus nombres. En un entorno local, colócalos en `data/`. Para el dataset realista usa la copia de la subcarpeta `resultados` original.

In [ ]:
import pandas as pd

ARCHIVOS_REQUERIDOS = ['dataset_entrenamiento_completo.csv']
def localizar(nombre):
    candidatos = [DATA_DIR / nombre, DATA_DIR / 'resultados' / nombre]
    return next((p for p in candidatos if p.is_file()), candidatos[0])

pendientes = [n for n in ARCHIVOS_REQUERIDOS if not localizar(n).is_file()]
if pendientes:
    print('Archivos requeridos:', pendientes)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Coloca los CSV en ' + str(DATA_DIR))
    cargados = files.upload()
    for nombre, contenido in cargados.items():
        if Path(nombre).name != nombre or nombre not in ARCHIVOS_REQUERIDOS:
            raise ValueError('Archivo inesperado: ' + nombre)
        (DATA_DIR / nombre).write_bytes(contenido)
    del cargados

DATA_FILES = {n: localizar(n) for n in ARCHIVOS_REQUERIDOS}
if any(not p.is_file() for p in DATA_FILES.values()):
    raise FileNotFoundError('Faltan CSV; repite esta celda y carga los indicados.')

# Inspección visible: dimensiones, tipos, valores faltantes y balance de clases.
for nombre, ruta in DATA_FILES.items():
    tabla = pd.read_csv(ruta)
    print('\n', nombre, '| filas y columnas:', tabla.shape)
    display(tabla.head())
    display(pd.DataFrame({'tipo': tabla.dtypes.astype(str), 'nulos': tabla.isna().sum()}))
    if 'clase' in tabla.columns:
        display(tabla['clase'].value_counts(dropna=False).rename('filas').to_frame())

REGISTRO = dict(experimento=EXPERIMENTO, parametros=CONFIG, run_id=RUN_ID,
    input_sha256={n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in DATA_FILES.items()},
    versiones={p: importlib.metadata.version(p) for p in DEPENDENCIAS.values()},
    estado='iniciado')
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
BASE_DIR = str(OUTPUT_DIR)
FILE_COMBINADO = str(DATA_FILES['dataset_entrenamiento_completo.csv'])
FILE_CSV_SPATIAL = str(OUTPUT_DIR / 'validacion_espacial_metricas.csv')
FILE_PNG_SPATIAL = str(OUTPUT_DIR / 'comparativa_validacion_espacial.png')
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])


### Definición: `FeatureSelfAttention`
Esta definición queda visible para revisar su arquitectura y proponer variantes. Sus valores originales se conservan.

In [ ]:
class FeatureSelfAttention(nn.Module):
    def __init__(self, input_dim):
        super(FeatureSelfAttention, self).__init__()
        self.query = nn.Linear(input_dim, input_dim, bias=False)
        self.key = nn.Linear(input_dim, input_dim, bias=False)
        self.value = nn.Linear(input_dim, input_dim, bias=False)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        attention_scores = torch.matmul(q.unsqueeze(2), k.unsqueeze(1)) / np.sqrt(x.shape[1])
        attention_weights = self.softmax(attention_scores.mean(dim=2))
        attended_x = v * attention_weights
        return attended_x, attention_weights


### Definición: `FireAttentionMLP`
Esta definición queda visible para revisar su arquitectura y proponer variantes. Sus valores originales se conservan.

In [ ]:
class FireAttentionMLP(nn.Module):
    def __init__(self, input_dim):
        super(FireAttentionMLP, self).__init__()
        self.attention = FeatureSelfAttention(input_dim)
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attended_x, weights = self.attention(x)
        output = self.classifier(attended_x)
        return output, weights


## 4. Desarrollo del experimento
Las siguientes celdas contienen el cuerpo del análisis original, con comentarios y código editable. Puedes inspeccionar las variables entre pasos. Ejecuta nuevamente desde la carga de datos si cambias una transformación previa.

### Etapa 1: Partición y preparación de muestras
La división aleatoria usa una semilla fija. El escalador se ajusta solo sobre entrenamiento. Una partición aleatoria no demuestra transferencia geográfica.

In [ ]:
print("="*75)
print(" EXPERIMENTO 2: VALIDACIÓN CRUZADA ESPACIAL POR BLOQUES PROVINCIALES")
print("="*75)

if not os.path.exists(FILE_COMBINADO):
    print(f"[ERROR] No se encuentra el archivo: {FILE_COMBINADO}")
    raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
df = pd.read_csv(FILE_COMBINADO)
features = ['elevacion', 'pendiente', 'orientacion', 'ndvi', 'evi', 'nbr', 'temp_c', 'precip_mm']

# Asignar provincia simulada por bloques de coordenadas latitudinales (10 provincias andinas)
# Como el CSV combinó presencias y ausencias, creamos 10 bloques espaciales latitudinales (provincias)
df['bloque_espacial'] = pd.qcut(df['elevacion'] + df['temp_c']*100, q=5, labels=['Bloque_Sur', 'Bloque_CentroSur', 'Bloque_Centro', 'Bloque_CentroNorte', 'Bloque_Norte'])

print(f"[INFO] Dataset cargado: {len(df)} registros.")
print(f"[INFO] Bloques espaciales para validación independiente:")
print(df['bloque_espacial'].value_counts())

# EXPERIMENTO 1: División Aleatoria Tradicional (80/20)
print("\n--- EJECUTANDO EXPERIMENTO 1 (División Aleatoria Tradicional 80/20) ---")
from sklearn.model_selection import train_test_split
X_all = df[features].values
y_all = df['clase'].values

X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(X_all, y_all, test_size=0.2, random_state=CONFIG['seed'], stratify=y_all)

# RF Aleatorio
rf_rand = RandomForestClassifier(n_estimators=CONFIG['rf_trees'], max_depth=CONFIG['rf_depth'], random_state=CONFIG['seed'], n_jobs=-1)


### Etapa 2: Entrenamiento de la red
Se optimizan los pesos con los datos de entrenamiento. El registro muestra la pérdida de la última tanda; no es una métrica de validación. Mantén fijo el conjunto de prueba al comparar configuraciones.

In [ ]:
rf_rand.fit(X_tr_rand, y_tr_rand)
acc_rf_rand = accuracy_score(y_te_rand, rf_rand.predict(X_te_rand))
auc_rf_rand = roc_auc_score(y_te_rand, rf_rand.predict_proba(X_te_rand)[:, 1])

# Attention MLP Aleatorio
scaler_rand = StandardScaler()
X_tr_rand_s = scaler_rand.fit_transform(X_tr_rand)
X_te_rand_s = scaler_rand.transform(X_te_rand)

nn_rand = FireAttentionMLP(input_dim=len(features))
opt_rand = optim.Adam(nn_rand.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-5)
crit = nn.BCELoss()

ds_rand = TensorDataset(torch.FloatTensor(X_tr_rand_s), torch.FloatTensor(y_tr_rand).unsqueeze(1))
ld_rand = DataLoader(ds_rand, batch_size=CONFIG['batch_size'], shuffle=True)

nn_rand.train()
for epoca in range(CONFIG['epochs']):
    for bx, by in ld_rand:
        opt_rand.zero_grad()
        out, _ = nn_rand(bx)
        loss = crit(out, by)
        loss.backward()
        opt_rand.step()
    if epoca == 0 or (epoca + 1) % 10 == 0 or epoca + 1 == CONFIG['epochs']:
        print(f"Época {epoca + 1}/{CONFIG['epochs']} | pérdida última tanda: {float(loss.detach()):.5f}")


### Etapa 3: Partición y preparación de muestras
La división aleatoria usa una semilla fija. El escalador se ajusta solo sobre entrenamiento. Una partición aleatoria no demuestra transferencia geográfica.

In [ ]:

nn_rand.eval()
with torch.no_grad():
    out_te_rand, _ = nn_rand(torch.FloatTensor(X_te_rand_s))
    pred_te_rand = (out_te_rand.numpy() >= 0.5).astype(int)
    acc_nn_rand = accuracy_score(y_te_rand, pred_te_rand)
    auc_nn_rand = roc_auc_score(y_te_rand, out_te_rand.numpy())

print(f"Random Forest (Aleatorio 80/20)   -> Accuracy: {acc_rf_rand:.4f} | AUC: {auc_rf_rand:.4f}")
print(f"Attention Net (Aleatorio 80/20)   -> Accuracy: {acc_nn_rand:.4f} | AUC: {auc_nn_rand:.4f}")

# EXPERIMENTO 2: Validación Espacial por Bloques Independientes (Spatial CV)
print("\n--- EJECUTANDO EXPERIMENTO 2 (Validación Espacial Independiente por Bloques) ---")

# Dejamos 1 bloque entero de prueba (un territorio completo que nunca vio el modelo)
test_bloque = 'Bloque_Norte'
train_df = df[df['bloque_espacial'] != test_bloque]
test_df = df[df['bloque_espacial'] == test_bloque]

X_train_sp = train_df[features].values
y_train_sp = train_df['clase'].values
X_test_sp = test_df[features].values
y_test_sp = test_df['clase'].values

# RF Espacial
rf_sp = RandomForestClassifier(n_estimators=CONFIG['rf_trees'], max_depth=CONFIG['rf_depth'], random_state=CONFIG['seed'], n_jobs=-1)


### Etapa 4: Entrenamiento de la red
Se optimizan los pesos con los datos de entrenamiento. El registro muestra la pérdida de la última tanda; no es una métrica de validación. Mantén fijo el conjunto de prueba al comparar configuraciones.

In [ ]:
rf_sp.fit(X_train_sp, y_train_sp)
pred_rf_sp = rf_sp.predict(X_test_sp)
proba_rf_sp = rf_sp.predict_proba(X_test_sp)[:, 1]

acc_rf_sp = accuracy_score(y_test_sp, pred_rf_sp)
auc_rf_sp = roc_auc_score(y_test_sp, proba_rf_sp)

# Attention Net Espacial
scaler_sp = StandardScaler()
X_tr_sp_s = scaler_sp.fit_transform(X_train_sp)
X_te_sp_s = scaler_sp.transform(X_test_sp)

nn_sp = FireAttentionMLP(input_dim=len(features))
opt_sp = optim.Adam(nn_sp.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-5)

ds_sp = TensorDataset(torch.FloatTensor(X_tr_sp_s), torch.FloatTensor(y_train_sp).unsqueeze(1))
ld_sp = DataLoader(ds_sp, batch_size=CONFIG['batch_size'], shuffle=True)

nn_sp.train()
for epoca in range(CONFIG['epochs']):
    for bx, by in ld_sp:
        opt_sp.zero_grad()
        out, _ = nn_sp(bx)
        loss = crit(out, by)
        loss.backward()
        opt_sp.step()
    if epoca == 0 or (epoca + 1) % 10 == 0 or epoca + 1 == CONFIG['epochs']:
        print(f"Época {epoca + 1}/{CONFIG['epochs']} | pérdida última tanda: {float(loss.detach()):.5f}")


### Etapa 5: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:

nn_sp.eval()
with torch.no_grad():
    out_te_sp, weights_sp = nn_sp(torch.FloatTensor(X_te_sp_s))
    pred_te_sp = (out_te_sp.numpy() >= 0.5).astype(int)
    acc_nn_sp = accuracy_score(y_test_sp, pred_te_sp)
    auc_nn_sp = roc_auc_score(y_test_sp, out_te_sp.numpy())

print(f"Random Forest (Validación Espacial) -> Accuracy: {acc_rf_sp:.4f} | AUC: {auc_rf_sp:.4f}")
print(f"Attention Net (Validación Espacial) -> Accuracy: {acc_nn_sp:.4f} | AUC: {auc_nn_sp:.4f}")

# Guardar métricas comparativas
comp_metrics = pd.DataFrame({
    'Estrategia de Evaluación': ['Exp 1: Aleatoria (80/20)', 'Exp 1: Aleatoria (80/20)', 'Exp 2: Espacial por Bloques', 'Exp 2: Espacial por Bloques'],
    'Modelo': ['Random Forest', 'Attention Net (PyTorch)', 'Random Forest', 'Attention Net (PyTorch)'],
    'Accuracy': [acc_rf_rand, acc_nn_rand, acc_rf_sp, acc_nn_sp],
    'ROC-AUC': [auc_rf_rand, auc_nn_rand, auc_rf_sp, auc_nn_sp]
})
comp_metrics.to_csv(FILE_CSV_SPATIAL, index=False)
print(f"\n[OK] Métricas de validación espacial guardadas en: {FILE_CSV_SPATIAL}")

# Graficar comparativa de experimentos
plt.figure(figsize=(10, 6), dpi=300)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

x = np.arange(2)


### Etapa 6: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:
width = 0.35

rf_accs = [acc_rf_rand*100, acc_rf_sp*100]
nn_accs = [acc_nn_rand*100, acc_nn_sp*100]

plt.bar(x - width/2, rf_accs, width, label='Random Forest', color='#d95f0e')
plt.bar(x + width/2, nn_accs, width, label='Attention Net (PyTorch)', color='#2b8cbe')

plt.ylabel('Exactitud / Accuracy (%)', fontsize=12, fontweight='bold')
plt.title('Comparación de Evaluación: Muestreo Aleatorio vs. Validación Espacial Independiente', fontsize=13, fontweight='bold', pad=15)
plt.xticks(x, ['Exp 1: Muestreo Aleatorio (80/20)', 'Exp 2: Validación Espacial (Spatial Block)'], fontsize=11, fontweight='semibold')
plt.ylim(50, 105)
plt.legend(loc='lower right', fontsize=11)

for i in range(len(x)):
    plt.text(x[i] - width/2, rf_accs[i] + 1, f"{rf_accs[i]:.1f}%", ha='center', fontweight='bold')
    plt.text(x[i] + width/2, nn_accs[i] + 1, f"{nn_accs[i]:.1f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(FILE_PNG_SPATIAL, bbox_inches='tight')
mostrar_figura(FILE_PNG_SPATIAL)
plt.close()
print(f"[OK] Gráfico comparativo guardado en: {FILE_PNG_SPATIAL}")
print("="*75)


## 5. Inspeccionar y guardar resultados
Las tablas se muestran debajo. El registro identifica parámetros, versiones y hashes. Esta celda marca el registro como completado: ejecútala solo cuando hayan terminado todas las etapas anteriores.

In [ ]:
for ruta in sorted(OUTPUT_DIR.glob('*.csv')):
    print(ruta.name)
    display(pd.read_csv(ruta).head(20))
REGISTRO['estado'] = 'completado'
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')
print('Salidas:', [p.name for p in sorted(OUTPUT_DIR.iterdir())])


In [ ]:
# Descarga las salidas antes de cerrar una sesión de Colab.
import shutil
paquete = shutil.make_archive(str(OUTPUT_DIR.parent / (EXPERIMENTO + '-' + RUN_ID)), 'zip', OUTPUT_DIR)
try:
    from google.colab import files
except ImportError:
    print('ZIP guardado en:', paquete)
else:
    files.download(paquete)


## 6. Proponer una colaboración

1. Plantea una pregunta concreta: por ejemplo, el efecto de un hiperparámetro o de una variable.
2. Conserva una ejecución con la configuración original y luego cambia una sola decisión.
3. Registra el hash del dataset, la partición, la semilla y todas las métricas relevantes.
4. En GitHub, abre un Issue para discutir la propuesta. Trabaja en un fork o en una rama
   con permisos y envía un pull request que explique el cambio y su evidencia.
5. Antes del commit, borra las salidas del cuaderno y revisa que no incluyan datos privados.
   Adjunta resultados únicamente cuando tengas permiso para compartirlos.

Edita la siguiente celda Markdown para documentar tu propuesta. No se envía nada automáticamente.

### Registro de la propuesta

- **Autor y objetivo:**
- **Hipótesis:**
- **Parámetro o código modificado:**
- **Dataset y hash:**
- **Semilla y partición:**
- **Resultado original y resultado de la variante:**
- **Limitaciones y conclusión:**
